# 32-Slice Polyp Full Workflow

This notebook documents the full HistoSeg 3D workflow used for a same-sample
32-slice Xenium polyp reconstruction with 2,785,128 aligned cells. It is a
pre-executed tutorial: the expensive reconstruction and gene-mapping steps were
run locally on a workstation, while ReadTheDocs only renders the saved outputs.

```{important}
RTD does **not** execute this notebook. The repository stores lightweight PNG
previews and summary tables, but not raw Xenium folders, `.h5ad` files, parquet
cell tables, or large interactive HTML files.
```


## 1. Environment

Install HistoSeg with the 3D and documentation extras in an editable checkout:


In [ ]:
%%bash
pip install -e ".[threed,docs]"


The 3D stack reader uses GitHub `pyXenium`. On Windows, long path support can be
needed when installing directly from GitHub:


In [ ]:
%%powershell
$env:GIT_CONFIG_COUNT='1'
$env:GIT_CONFIG_KEY_0='core.longpaths'
$env:GIT_CONFIG_VALUE_0='true'
python -m pip install "pyXenium @ git+https://github.com/hutaobo/pyXenium.git"


Hardware guidance for the full 32-slice run:

| Resource | Recommendation |
| --- | --- |
| RAM | 64 GB or more for the full 2.7M-cell workflow |
| Disk | Keep raw Xenium outputs and HistoSeg outputs on fast local or network storage |
| Runtime | Run locally or on a workstation/HPC node, not inside RTD |
| Visualization | Generate interactive HTML locally; commit only PNG previews to docs |


In [1]:
from pathlib import Path

xenium_root = Path(r"Y:/long/spatialpathologist/3D aligment/polyp")
stack_root = xenium_root / "histoseg_3d_reconstruction"
merged_h5ad = xenium_root / "pdc_merge_leiden/polyp_32samples_min3_count5_leiden_20260501_processed_leiden.h5ad"
segmentation_strategy = xenium_root / "contour for alignment/segmentationstrategy.txt"

print("Workflow root:", xenium_root)
print("Output root:", stack_root)
print("Merged AnnData:", merged_h5ad.name)
print("Cluster source: leiden_1_0")


Workflow root: Y:/long/spatialpathologist/3D aligment/polyp
Output root: Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction
Merged AnnData: polyp_32samples_min3_count5_leiden_20260501_processed_leiden.h5ad
Cluster source: leiden_1_0


## 2. Data Organization

The manuscript-scale run used 32 Xenium folders from one polyp sample. HistoSeg
discovers them in numeric slice order with `--sample-glob A079-C-008_*`.

```text
polyp/
├── A079-C-008_1/
├── A079-C-008_2/
├── ...
├── A079-C-008_16/
├── A079-C-008_25/
├── ...
├── A079-C-008_40/
├── contour for alignment/
│   └── segmentationstrategy.txt
├── pdc_merge_leiden/
│   └── polyp_32samples_min3_count5_leiden_20260501_processed_leiden.h5ad
└── histoseg_3d_reconstruction/
```

The segmentation strategy maps Leiden clusters to five named structures, one
line per structure:

```text
18
31,3,30
1,6,8
24,0,2,16,11,13
27,22,29,26,28,25,12,21,23,7,17,19,5,14,15,4,9,10,20
```

Those cluster IDs come from the merged AnnData column `leiden_1_0`. Per-slice
Xenium graphclust labels are not interchangeable with this strategy.


## 3. 3D Contour Reconstruction

Run the full stack reconstruction with `histoseg-3d reconstruct-stack`. This
step reads the Xenium slices with `pyXenium`, builds 2D contours, aligns each
neighboring pair, samples the aligned contours into 3D, and reconstructs
per-structure surfaces with voxelization, 3D smoothing, and Marching Cubes.


In [2]:
%%bash
histoseg-3d reconstruct-stack \
  --xenium-root "Y:/long/spatialpathologist/3D aligment/polyp" \
  --segmentation-strategy "Y:/long/spatialpathologist/3D aligment/polyp/contour for alignment/segmentationstrategy.txt" \
  --merged-h5ad "Y:/long/spatialpathologist/3D aligment/polyp/pdc_merge_leiden/polyp_32samples_min3_count5_leiden_20260501_processed_leiden.h5ad" \
  --merged-cluster-column leiden_1_0 \
  --out-dir "Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction" \
  --z-spacing-um 5 \
  --hard-alignment-maxiter 40 \
  --voxel-size-um 80 \
  --point-sample-distance-um 80 \
  --mesh-smoothing-sigma-um 40 \
  --mesh-export-formats ply,obj \
  --no-alignment-preview \
  --overwrite


HistoSeg 3D reconstruction complete.
Aligned slices: 32
Pairwise alignments: 31
Hard alignments accepted: 31 / 31
Soft TPS alignments accepted: 31 / 31
Wrote: histoseg_3d_reconstruction/3d_stack_reconstruction_summary.json


Final reconstruction QC from this run:

| Metric | Value |
| --- | ---: |
| Aligned slices | 32 |
| Pairwise alignments | 31 |
| Hard alignments accepted | 31 / 31 |
| Soft TPS alignments accepted | 31 / 31 |
| Mean raw pairwise union IoU | 0.7220 |
| Mean hard-aligned union IoU | 0.9386 |
| Mean soft-aligned union IoU | 0.9480 |
| Minimum soft-aligned union IoU | 0.9136 |
| TPS landmarks per pair | 1,660 |
| 3D sampled contour points | 94,108 |
| Reconstructed structure meshes | 5 |
| Watertight structure meshes | 5 / 5 |

![32-slice 3D contour stack preview](../_static/threed/polyp/3d_stack_preview.png)


The main reconstruction artifacts are:

```text
xenium_slice_manifest.csv
slice_contours/*/xenium_explorer_annotations.geojson
aligned_contours/*_aligned.geojson
pairwise_alignment_metrics.csv
aligned_slice_manifest.csv
aligned_contour_3d_points.csv
histoseg_3d_contour_stack.html
meshes/Structure_1.ply ... meshes/Structure_5.ply
meshes/Structure_1.obj ... meshes/Structure_5.obj
meshes/mesh_manifest.csv
meshes/mesh_qc_summary.json
3d_stack_reconstruction_summary.json
```


## 4. Cell Projection And 3D Cell Cloud

After contour alignment, project all cells from the merged AnnData into the same
3D coordinate system. The command below writes an aligned cell table that can be
reused by gene mapping and visualization steps.


In [3]:
%%bash
histoseg-3d project-cells \
  --h5ad "Y:/long/spatialpathologist/3D aligment/polyp/pdc_merge_leiden/polyp_32samples_min3_count5_leiden_20260501_processed_leiden.h5ad" \
  --stack-root "Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction" \
  --out-parquet "Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction/aligned_leiden_3d_cells.parquet" \
  --sample-column sample_id \
  --barcode-column barcode \
  --x-column x_centroid \
  --y-column y_centroid \
  --label-columns leiden_1_0 \
  --write-cache


Projected 2,785,128 cells into the 3D reconstruction coordinate system.
Slices represented: 32
Z range: 0.0 to 155.0 um
Wrote: aligned_leiden_3d_cells.parquet


The interactive 300k-cell Plotly view is useful locally, but it is intentionally
not committed to RTD because it is a large HTML artifact.


In [4]:
%%bash
histoseg-3d render-cell-cloud \
  --stack-root "Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction" \
  --aligned-cells-parquet "Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction/aligned_leiden_3d_cells.parquet" \
  --out-html "Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction/leiden_3d_cells_300k.html" \
  --label-column leiden_1_0 \
  --max-points 300000


Wrote local interactive HTML: leiden_3d_cells_300k.html
Sampled cells shown: 300,000
The HTML is kept outside the documentation repository.


Static preview of the same 300k-cell sample. The Z axis is visually scaled in
this PNG so the 32 slice layers are visible in a flat tissue stack.

![300k aligned Leiden 3D cell cloud preview](../_static/threed/polyp/32slice/leiden_3d_cell_cloud_300k_preview.png)


## 5. Gene Spatial Modules

The downstream analysis maps genes into normalized 3D enrichment fields,
extracts nested hotspot surfaces, quantifies voxel-based overlap and signed
distances to each structure, and clusters genes by 3D localization.


In [5]:
%%bash
histoseg-3d discover-spatial-modules \
  --h5ad "Y:/long/spatialpathologist/3D aligment/polyp/pdc_merge_leiden/polyp_32samples_min3_count5_leiden_20260501_processed_leiden.h5ad" \
  --aligned-cells-parquet "Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction/aligned_leiden_3d_cells.parquet" \
  --stack-root "Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction" \
  --out-dir "Y:/long/spatialpathologist/3D aligment/polyp/histoseg_3d_reconstruction/gene_overlays/batch_3d_genes_starter_panel" \
  --genes GREM1 COL1A1 COL1A2 ACTA2 PDGFRA TAGLN FAP EPCAM MUC2 OLFM4 LGR5 MKI67 PTPRC CD3D CD3E CD8A CD4 MS4A1 CD79A LYZ CD68 C1QA PECAM1 RGS5 \
  --mesh-export-formats ply


Batch spatial module discovery complete.
Genes requested: 24
Genes completed: 24
Hotspot tiers: top15, top10, top05
Wrote gene-structure matrices and clustered heatmaps.


For each gene, HistoSeg computes normalized enrichment:

```text
smoothed gene expression sum / smoothed total cell count
```

This avoids mistaking high cellularity for high expression. The GREM1 example
below is shown as nested Marching Cubes hotspot surfaces rather than as a dense
single-cell point cloud.

![GREM1 nested 3D hotspot surfaces](../_static/threed/polyp/24gene/GREM1_nested_3d_hotspot_surfaces.png)

![GREM1 Structure 5 focus](../_static/threed/polyp/24gene/GREM1_structure5_focus.png)

For this run, the GREM1 top 5% hotspot core is concentrated in Structure 5:

| Hotspot | Structure | Fraction inside structure | Median signed distance |
| --- | --- | ---: | ---: |
| GREM1 top05 | Structure 5 | 0.8621 | -35.0 um |
| GREM1 top05 | Structure 3 | 0.1169 | 25.0 um |
| GREM1 top15 | Structure 5 | 0.7356 | -20.0 um |
| GREM1 top15 | Structure 3 | 0.3003 | 15.0 um |

Negative signed distance means the hotspot is physically embedded inside the
structure mask. Positive values mean it is outside the structure boundary.


The 24-gene panel produces compact gene-by-structure matrices. These are small
enough to commit as figures and review in RTD.

![Top 5% fraction-inside spatial modules](../_static/threed/polyp/24gene/fraction_inside_top05_spatial_clustermap.png)

![Top 5% signed-distance spatial modules](../_static/threed/polyp/24gene/signed_distance_top05_spatial_clustermap.png)

Interpretation from the 24-gene starter panel:

- `GREM1`, `FAP`, `TAGLN`, `ACTA2`, and `COL1A1/COL1A2` localize strongly to
  Structure 5, consistent with a stromal/fibrotic compartment.
- `MUC2`, `OLFM4`, and `MKI67` are enriched around Structure 3, consistent with
  a differentiated or proliferative epithelial compartment.
- `LGR5` and `EPCAM` are prominent around Structure 4, consistent with an
  epithelial or stem-like niche.
- `C1QA` and `PDGFRA` show a Structure 2 signal, suggesting a macrophage or
  perivascular-associated compartment.


## 6. Reproducibility And Re-entry Points

The full pipeline is intentionally split into reusable artifacts. If a run is
interrupted or a figure needs to be regenerated, restart from the nearest saved
artifact rather than recomputing the whole stack.

| Task | Re-entry command | Main inputs |
| --- | --- | --- |
| Rebuild 3D contours and meshes | `histoseg-3d reconstruct-stack` | Xenium slice folders, segmentation strategy, merged `.h5ad` |
| Reproject all cells | `histoseg-3d project-cells` | `aligned_slice_manifest.csv`, pairwise transforms, merged `.h5ad` |
| Render local cell HTML | `histoseg-3d render-cell-cloud` | `aligned_leiden_3d_cells.parquet` |
| Batch-map genes | `histoseg-3d discover-spatial-modules` | aligned cells parquet, meshes, merged `.h5ad` |
| Recompute one gene relationship table | `histoseg-3d quantify-gene-structure` | one gene density folder and cached structure masks |
| Regenerate clustermap figures | `histoseg-3d plot-spatial-modules` | batch gene matrix CSVs |

Large local artifacts deliberately excluded from RTD include:

```text
raw Xenium output folders
merged AnnData .h5ad
aligned_leiden_3d_cells.parquet
leiden_3d_cells_300k.html
per-gene voxel CSVs
per-gene PLY/OBJ isosurfaces
```

Commit only lightweight figures and summary text to RTD. This keeps the tutorial
reviewable while preserving the complete local commands needed to reproduce the
manuscript-scale workflow.
